In [2]:
import torch as t
import torch.nn as nn

In [3]:
class RPE(nn.Module):
    def __init__(self, heads: int, seq_len: int, max_distance: int, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # shape = (distance buckets, number of heads)
        self.rpe_bias_table = nn.Embedding(max_distance+1, heads)

        positions = t.arange(seq_len)
        distances = positions.unsqueeze(1) - positions.unsqueeze(0) # shape = (L, L)
        
        c_dist = t.clamp(distances, min=0, max=max_distance) # automatically makes the negative become zero. 
        
        self.register_buffer("c_dist_cache", c_dist)

    def forward(self, x):
        
        cur_len = x.size(-1)
        c_dist = self.c_dist_cache[:cur_len, :cur_len]

        b: t.Tensor = self.rpe_bias_table(c_dist)  # dim: (L, L, n)

        rpe_bias = b.permute(2, 0, 1).unsqueeze(0) # dim: (1, n, L, L)

        return x + rpe_bias

        

In [4]:
rpe = RPE(8, 1024, 8)
input = t.randn(4, 8, int(1024/8), int(1024/8))
output = rpe(input)

In [5]:
input

tensor([[[[-6.8060e-01, -1.7060e+00, -9.8761e-01,  ..., -5.8048e-02,
           -5.8977e-01, -9.8559e-01],
          [-1.4182e+00, -1.5647e+00, -1.8298e+00,  ..., -1.4849e+00,
            7.6467e-01,  3.8282e-01],
          [-5.7557e-02,  2.1495e+00, -7.2320e-01,  ..., -7.4461e-02,
            4.2912e-01,  1.0174e+00],
          ...,
          [-1.1119e-01,  7.5437e-01,  1.5193e+00,  ...,  1.8478e-01,
            2.7798e+00,  1.0275e+00],
          [ 6.2086e-01, -1.9666e+00, -1.6650e-01,  ...,  6.6304e-01,
            9.0736e-02,  1.0531e+00],
          [-7.9587e-01, -2.2823e+00,  6.4551e-01,  ..., -4.2920e-01,
           -8.9868e-01,  6.9274e-01]],

         [[-1.1829e+00, -1.7275e-01,  5.5228e-01,  ..., -9.9821e-01,
           -1.7256e-01,  1.1590e+00],
          [-1.5045e+00,  7.4106e-01, -8.0726e-01,  ..., -1.2806e+00,
           -1.9377e+00, -9.4633e-01],
          [-7.3278e-01,  6.6724e-02,  9.5756e-01,  ...,  1.5974e-01,
           -1.2832e-01, -8.7692e-02],
          ...,
     

In [6]:
output

tensor([[[[-3.1536e+00, -4.1790e+00, -3.4606e+00,  ..., -2.5310e+00,
           -3.0628e+00, -3.4586e+00],
          [-2.0733e+00, -4.0377e+00, -4.3028e+00,  ..., -3.9579e+00,
           -1.7083e+00, -2.0902e+00],
          [-6.5555e-01,  1.4943e+00, -3.1962e+00,  ..., -2.5475e+00,
           -2.0439e+00, -1.4556e+00],
          ...,
          [-9.1611e-01, -5.0552e-02,  7.1439e-01,  ..., -2.2882e+00,
            3.0677e-01, -1.4455e+00],
          [-1.8406e-01, -2.7715e+00, -9.7142e-01,  ...,  7.8455e-03,
           -2.3823e+00, -1.4199e+00],
          [-1.6008e+00, -3.0872e+00, -1.5942e-01,  ..., -1.0272e+00,
           -1.5539e+00, -1.7803e+00]],

         [[-4.3562e-01,  5.7454e-01,  1.2996e+00,  ..., -2.5092e-01,
            5.7473e-01,  1.9063e+00],
          [-1.1294e+00,  1.4883e+00, -5.9969e-02,  ..., -5.3327e-01,
           -1.1904e+00, -1.9904e-01],
          [-1.2578e+00,  4.4183e-01,  1.7048e+00,  ...,  9.0703e-01,
            6.1897e-01,  6.5960e-01],
          ...,
     

In [7]:
positions = t.arange(6)

rpe_bias_table = nn.Embedding(4, 2)

distances = positions.unsqueeze(1) - positions.unsqueeze(0)
c_dist = t.clamp(distances, min=0, max=3)

b = rpe_bias_table(c_dist)

In [8]:
c_dist

tensor([[0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0],
        [2, 1, 0, 0, 0, 0],
        [3, 2, 1, 0, 0, 0],
        [3, 3, 2, 1, 0, 0],
        [3, 3, 3, 2, 1, 0]])

In [9]:
b.permute(2, 0, 1)

tensor([[[-1.4412e-01, -1.4412e-01, -1.4412e-01, -1.4412e-01, -1.4412e-01,
          -1.4412e-01],
         [ 1.6179e+00, -1.4412e-01, -1.4412e-01, -1.4412e-01, -1.4412e-01,
          -1.4412e-01],
         [ 7.0957e-01,  1.6179e+00, -1.4412e-01, -1.4412e-01, -1.4412e-01,
          -1.4412e-01],
         [ 2.1659e-03,  7.0957e-01,  1.6179e+00, -1.4412e-01, -1.4412e-01,
          -1.4412e-01],
         [ 2.1659e-03,  2.1659e-03,  7.0957e-01,  1.6179e+00, -1.4412e-01,
          -1.4412e-01],
         [ 2.1659e-03,  2.1659e-03,  2.1659e-03,  7.0957e-01,  1.6179e+00,
          -1.4412e-01]],

        [[ 2.3808e+00,  2.3808e+00,  2.3808e+00,  2.3808e+00,  2.3808e+00,
           2.3808e+00],
         [-9.6030e-01,  2.3808e+00,  2.3808e+00,  2.3808e+00,  2.3808e+00,
           2.3808e+00],
         [-4.9346e-01, -9.6030e-01,  2.3808e+00,  2.3808e+00,  2.3808e+00,
           2.3808e+00],
         [ 7.9227e-01, -4.9346e-01, -9.6030e-01,  2.3808e+00,  2.3808e+00,
           2.3808e+00],
        

In [10]:
b.permute(2, 0, 1).unsqueeze(0).shape

torch.Size([1, 2, 6, 6])

In [11]:
from typing import Literal
import math
import torch.nn.functional as F

class ALiBi(nn.Module):
    def __init__(self, heads: int, seq_len: int, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.seq_len = seq_len

        positions = t.arange(seq_len, dtype=t.float32)
        distances = positions.unsqueeze(1) - positions.unsqueeze(0)

        # negative so that the penalties remain negative.
        penals = -t.abs(distances)

        slopes = 2 ** -(t.arange(heads, dtype=t.float32) * 8 / heads)
        slopes = slopes.reshape(heads, 1, 1)

        self.register_buffer("alibi_bias", slopes * penals)

    def forward(self, x):
        causal_mask = t.triu(
            t.full((self.seq_len, self.seq_len), float("-inf")), diagonal=1
        )
        final_attn_mask = self.alibi_bias + causal_mask

        return x + final_attn_mask

class RPE(nn.Module):
    def __init__(self, heads: int, seq_len: int, max_distance: int, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # shape = (distance buckets, number of heads)
        self.rpe_bias_table = nn.Embedding(max_distance + 1, heads)

        positions = t.arange(seq_len)
        distances = positions.unsqueeze(1) - positions.unsqueeze(0)  # shape = (L, L)

        c_dist = t.clamp(
            distances, min=0, max=max_distance
        )  # automatically makes the negative become zero.

        self.register_buffer("c_dist_cache", c_dist)

    def forward(self, x):

        cur_len = x.size(-1)
        c_dist = self.c_dist_cache[:cur_len, :cur_len]

        b: t.Tensor = self.rpe_bias_table(c_dist)  # dim: (L, L, n)

        rpe_bias = b.permute(2, 0, 1).unsqueeze(0)  # dim: (1, n, L, L)

        return x + rpe_bias

class RoPE(nn.Module):
    def __init__(self, dim: int, seq_len: int):
        super().__init__()

        # theta = 10000^{-2i/d} part
        self.theta = 10000 ** (-2 * t.arange(0, dim // 2) / dim).unsqueeze(
            -1
        )  # shape = [128, 1]

        # converting to e^{i m theta}
        self.e = t.exp(
            1j * t.arange(seq_len).unsqueeze(-1) @ self.theta.T
        )  # shape = [1024, 1] x [1, 128] = [1024, 128].

        self.e = self.e.unsqueeze(0)  # [1024, 128] -> [1, 1024, 128]

        self.register_buffer("e", self.e)

    def forward(self, xq, xk):

        # pairing them up along the embedding dimension: [1, 2, 3, 4] -> [[1, 2], [3, 4]]
        xq_paired = xq.reshape(*xq.shape[:-1], -1, 2)
        xk_paired = xk.reshape(*xq.shape[:-1], -1, 2)

        # view them as complex: [[1, 2], [3, 4]] -> [[1, 2j], [3, 4j]]
        xq_comp = t.view_as_complex(xq_paired)
        xk_comp = t.view_as_complex(xk_paired)

        # rotato.
        xq_rot = xq_comp * self.e
        xk_rot = xk_comp * self.e

        # back to real
        xq_out = t.view_as_real(xq_rot).flatten(-2)
        xk_out = t.view_as_real(xk_rot).flatten(-2)

        return xq_out, xk_out

In [12]:
class FlexibleAttentionBlock(nn.Module):
    def __init__(
        self,
        pe: Literal["RoPE", "ALiBi", "RPE"],
        variant: Literal["SWA", "MQA", "GQA"],
        dim,
        seq_len,
        heads,
        # some specific arguments for the variants smh.
        window_size: int | None = None,
        max_distance: int | None = None,
        groups: int | None = None,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        assert dim % heads == 0, "dim must be divisible by n_heads"

        self.pe = pe
        self.variant = variant

        self.dim = dim
        self.heads = heads  # no.of attention heads
        self.head_dim = dim // heads

        if self.pe == "RoPE":
            self.PE = RoPE(dim=self.head_dim, seq_len=seq_len)

        elif self.pe == "ALiBi":
            self.PE = ALiBi(heads=heads, seq_len=seq_len)

        elif self.pe == "RPE":
            assert (
                max_distance is not None
            ), "Pass in max_distance if you want to use RPE"
            self.PE = RPE(heads=heads, seq_len=seq_len, max_distance=max_distance)

        # masking part
        positions = t.arange(seq_len)

        if self.variant == "SWA":
            assert (
                window_size is not None
            ), "Nigga, pass in window_size if you using SWA."

            distances = positions.unsqueeze(1) - positions.unsqueeze(0)
            allowed = (distances >= 0) & (distances < window_size)
            mask = t.where(allowed, 0.0, float("-inf"))

            self.groups = self.heads
        else:
            # Standard causal mask for MQA / GQA so it doesn't throw an error
            mask = t.where(
                positions.unsqueeze(1) <= positions.unsqueeze(0), 0.0, float("-inf")
            )

        # Register as a buffer so PyTorch handles moving it to the GPU automatically
        self.register_buffer("attn_mask", mask)

        if self.variant == "MQA":
            self.groups = 1
            self.repeats = self.heads

        elif self.variant == "GQA":
            assert groups is not None, "pass in groups, you selected GQA."
            assert (
                heads % groups == 0
            ), "Number of heads must be divisible by the number of groups."
            self.groups = groups
            self.repeats = heads // groups

        self.Qw = nn.Linear(dim, dim)
        self.Kw = nn.Linear(dim, dim)
        self.Vw = nn.Linear(dim, dim)

        if self.variant in ["MQA", "GQA"]:
            self.Kw = nn.Linear(dim, self.groups * self.head_dim)
            self.Vw = nn.Linear(dim, self.groups * self.head_dim)

        self.Wo = nn.Linear(dim, dim)  # output projection.

    def forward(self, x: t.Tensor):

        B, L, _ = x.size()
        masked_attn = self.attn_mask[
            :L, :L
        ]  # slicing to the current length (only affects edge cases).

        Q = self.Qw(x)
        K = self.Kw(x)
        V = self.Vw(x)

        Q = Q.view(B, L, self.heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(B, L, self.groups, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(B, L, self.groups, self.head_dim).permute(0, 2, 1, 3)

        if self.pe == "RoPE":
            Q, K = self.PE(Q, K)

        if self.variant in ["MQA", "GQA"]:
            K = K.repeat_interleave(self.repeats, dim=1)
            V = V.repeat_interleave(self.repeats, dim=1)

        attn_weights = Q @ K.permute(
            0, 1, 3, 2
        )  # (B, n, L, E/n) x (B, n, E/n, L) = (B, n, L, L)

        # size = (B, n, L, L)
        scaled_attn_weights = attn_weights / math.sqrt(self.head_dim)

        if self.pe == "ALiBi":
            scaled_attn_weights = self.PE(scaled_attn_weights)
        elif self.pe == "RPE":
            scaled_attn_weights = self.PE(scaled_attn_weights)

        masked_attn = scaled_attn_weights + masked_attn

        attn_scores = F.softmax(masked_attn, dim=-1)

        # final output
        out = attn_scores @ V  # (B, n, L, L) x (B, n, L, E/n) = (B, n, L, E/n)
        out = out.permute(0, 2, 1, 3).contiguous().view(B, L, self.dim)  # (B, L, E)
        out = self.Wo(out)

        return out

In [13]:
attn = FlexibleAttentionBlock(pe="ALiBi", variant="GQA", dim=128, seq_len=1024, heads=8, groups=4)

In [14]:
test_input = t.randn(4, 1024, 128)
output = attn(test_input)

In [15]:
output

tensor([[[-6.9162e-02,  1.5436e-01, -3.7052e-01,  ..., -2.5382e-01,
          -2.5550e-01,  1.5644e-01],
         [-3.8484e-01, -5.7979e-01,  1.9344e-01,  ...,  4.6834e-02,
          -3.9209e-01,  4.7661e-01],
         [-2.7732e-01, -2.1057e-01,  3.5347e-01,  ...,  2.9865e-01,
          -5.2141e-01, -6.3399e-01],
         ...,
         [-4.3116e-01, -2.0329e-01,  3.3220e-01,  ...,  4.5895e-01,
          -3.5521e-02, -1.8961e-01],
         [ 1.7825e-01, -4.5583e-01,  4.0111e-01,  ...,  1.5334e-02,
          -3.8219e-01,  5.5070e-01],
         [ 5.0695e-01,  6.5375e-02, -1.5997e-02,  ...,  1.2572e-01,
          -2.7615e-01,  1.8317e-01]],

        [[-1.0527e-01, -7.0671e-02, -6.3345e-02,  ...,  2.7912e-01,
          -1.7550e-01, -2.2221e-01],
         [-3.0004e-01,  2.8788e-04, -3.3833e-01,  ...,  1.1894e-01,
          -1.8111e-01,  1.7488e-02],
         [-3.1714e-01, -1.2660e-01,  3.7256e-01,  ..., -8.9762e-02,
          -2.6139e-01, -4.2955e-01],
         ...,
         [ 3.9009e-01, -5

In [16]:
class MLP(nn.Module):
    def __init__(self, dim, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.fc1 = nn.Linear(dim, 4 * dim)
        self.fc2 = nn.Linear(4 * dim, dim)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)  # gaussian error linear unit.
        x = self.fc2(x)
        return x

In [17]:
class ConvAttentionBlock(nn.Module):
    def __init__(self, heads: int, seq_len: int, dim: int, kernel_size: int, padding: int, attention_block: nn.Module):
        super().__init__()
        
        self.conv1 = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=padding)
        self.norm1 = nn.LayerNorm(dim)
        self.attn = attention_block
        self.norm2 = nn.LayerNorm(dim)
        self.gelu = nn.GELU()

        self.mlp = MLP(dim)

    def forward(self, x):
        
        residue = x
        x = self.norm1(x)
        
        x = x.permute(0, 2, 1)  # (B, E, L)
        x = self.conv1(x)
        x = x.permute(0, 2, 1)  # (B, L, E)

        x = self.gelu(x)
        x = self.attn(x)
        
        x = x + residue
        
        residue = x
        
        x = self.norm2(x)

        x = self.mlp(x)
        x = x + residue
        
        return x

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, dim: int, kernel_size: int, padding: int):
        super().__init__()

        self.conv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=padding)
        self.norm1 = nn.LayerNorm(dim)
        self.gelu = nn.GELU()
        self.mlp = MLP(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        residue = x
        x = self.norm1(x)
        
        x = x.permute(0, 2, 1)  # (B, E, L)
        x = self.conv(x)
        x = x.permute(0, 2, 1)  # (B, L, E)

        x = self.gelu(x)

        x = x + residue

        x = self.norm2(x)
        x = self.mlp(x)

        x = x + residue

        return x
        

In [18]:
attn = FlexibleAttentionBlock(
            pe="ALiBi",
            variant="GQA",
            dim=256,
            seq_len=1028,
            heads=8,
            groups=4,
)
conv_attn = ConvAttentionBlock(heads=8, seq_len=1028, dim=256, kernel_size=9, padding=4, attention_block=attn)

In [19]:
output = conv_attn(t.randn(4, 1028, 256))

In [21]:
output

tensor([[[-1.5972, -0.4145, -1.3036,  ..., -0.0664, -2.3496, -0.9227],
         [-0.1556, -2.8576, -1.1039,  ..., -0.4276,  0.4507,  0.7879],
         [-0.2386, -0.7451, -0.1795,  ...,  0.4816, -0.0099,  1.5516],
         ...,
         [-0.3081,  0.4236,  0.8566,  ...,  1.5161, -1.2625, -0.5055],
         [-0.2647, -0.5588, -0.8069,  ...,  0.6940,  0.5551, -0.4370],
         [-2.1184, -0.1748,  0.7562,  ..., -0.6232,  0.8963, -0.4381]],

        [[ 0.0050,  1.2295, -1.0706,  ...,  0.3629, -0.8981,  1.1619],
         [ 1.4000,  2.0361, -1.5593,  ...,  2.4069,  0.1385, -0.6609],
         [ 0.7070, -0.9594,  0.0124,  ..., -0.6071,  1.7967,  1.1972],
         ...,
         [ 0.0169,  0.3029, -0.5428,  ...,  0.6395,  0.0619,  1.4570],
         [-2.0488, -1.5992,  0.5701,  ..., -2.4857, -0.1535,  0.0774],
         [ 0.3154, -0.9322,  0.2604,  ..., -0.7537,  0.6510, -0.0814]],

        [[-1.5936, -0.9631, -0.7963,  ..., -0.7300,  1.9185, -0.0069],
         [ 0.5428, -1.7444, -1.5857,  ..., -0